# ExportGuard — GPU Benchmark (Google Colab)

This notebook runs the **same data pipeline on CPU (pandas) vs GPU (cudf.pandas)**
and compares wall-clock time.

**How to use:**
1. Go to `Runtime → Change runtime type` → Select **T4 GPU**
2. Run all cells (`Runtime → Run all`)
3. Results appear at the bottom — CPU time vs GPU time

In [ ]:
# ── Step 1: Install RAPIDS (cudf + cuml) ──────────────────
!pip install cudf-pandas cuml -q
print("RAPIDS installed")

In [ ]:
# ── Step 2: Generate synthetic data inside Colab ───────────
# This creates a 500K-row dataset (enough for a clear speed difference)
# so you don't need to upload large files.

import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

N_SHIPMENTS = 500_000
N_COUNTRIES = 40
N_BUYERS = 2000
MONTHS = pd.date_range("2022-01-01", "2024-12-01", freq="MS")

print("Generating country risk data...")
country_rows = []
countries = [f"country_{i:03d}" for i in range(N_COUNTRIES)]
for country in countries:
    base_stab = np.clip(np.random.normal(0.6, 0.25), 0.05, 0.95)
    base_vol = np.clip(np.random.normal(0.3, 0.15), 0.01, 0.95)
    sanction = 1 if np.random.random() < 0.08 else 0
    for i, month in enumerate(MONTHS):
        walk = 0.05 * np.sin(2 * np.pi * i / 12)
        country_rows.append({
            "country": country,
            "month": month,
            "political_stability_score": round(np.clip(base_stab + walk + np.random.normal(0,0.03), 0, 1), 4),
            "currency_volatility_index": round(np.clip(base_vol + 0.5*walk + np.random.normal(0,0.02), 0, 1), 4),
            "trade_sanctions_flag": sanction,
        })
cr = pd.DataFrame(country_rows)
cr.to_csv(DATA_DIR / "country_risk.csv", index=False)
print(f"  {len(cr):,} rows")

print("Generating shipment data...")
buyer_ids = [f"buyer_{i:05d}" for i in range(N_BUYERS)]
buyer_country = np.random.choice(countries, size=N_BUYERS)
buyer_risk = np.random.choice(["low","medium","high"], size=N_BUYERS, p=[0.5,0.3,0.2])

records = []
for i in range(N_SHIPMENTS):
    bid = buyer_ids[np.random.randint(N_BUYERS)]
    bi = buyer_ids.index(bid)
    rt = buyer_risk[bi]
    bc = buyer_country[bi]
    
    if rt == "high":
        tp = [0.50, 0.35, 0.10, 0.04, 0.01]
    elif rt == "medium":
        tp = [0.15, 0.40, 0.30, 0.10, 0.05]
    else:
        tp = [0.05, 0.20, 0.35, 0.25, 0.15]
    term = np.random.choice(["advance","lc","credit_30","credit_60","credit_90"], p=tp)
    
    if term == "advance":
        delay = max(0, int(np.random.normal(0, 2)))
    elif term == "lc":
        delay = max(0, int(np.random.normal(15, 8)))
    elif term == "credit_30":
        delay = max(0, int(np.random.normal(30, 10)))
    elif term == "credit_60":
        delay = max(0, int(np.random.normal(60, 15)))
    else:
        delay = max(0, int(np.random.normal(90, 20)))
    
    if rt == "high":
        delay += 20
    elif rt == "medium":
        delay += 5
    
    dispute_prob = 0.01
    if rt == "high":
        dispute_prob = 0.12
    elif rt == "medium":
        dispute_prob = 0.04
    if delay > 120:
        dispute_prob += 0.10
    elif delay > 60:
        dispute_prob += 0.05
    disputed = np.random.random() < dispute_prob
    
    if disputed:
        pif = 0.20
    elif rt == "high":
        pif = 0.70
    elif rt == "medium":
        pif = 0.88
    else:
        pif = 0.97
    paid_full = np.random.random() < pif
    
    date = MONTHS[np.random.randint(len(MONTHS))] + pd.Timedelta(days=np.random.randint(0, 28))
    
    records.append({
        "shipment_id": f"SHP-{i:07d}",
        "exporter_id": f"exporter_{np.random.randint(600):04d}",
        "buyer_id": bid,
        "buyer_country": bc,
        "hs_code": int(np.random.randint(1, 22)),
        "product_category": np.random.choice([
            "textiles","chemicals","machinery","electronics","automotive",
            "pharmaceuticals","plastics","agricultural","steel","ceramics",
            "leather","paper","wood","minerals","food_processed","rubber","glass"
        ]),
        "shipment_date": date,
        "invoice_value_usd": round(np.clip(np.random.lognormal(mean=9.5, sigma=1.2), 100, 5_000_000), 2),
        "payment_terms": term,
        "payment_delay_days": delay,
        "was_disputed": int(disputed),
        "was_paid_in_full": int(paid_full),
    })
    if (i+1) % 100000 == 0:
        print(f"  {i+1:,} rows generated...")

df = pd.DataFrame(records)
df.to_csv(DATA_DIR / "shipments.csv", index=False)
print(f"  {len(df):,} rows saved")
print("Data generation complete!")

In [ ]:
# ── Step 3: Copy the transform.py into Colab ──────────────
# This creates the shared transform module locally.

%%writefile /content/transform.py
import pandas as pd
import numpy as np
from pathlib import Path

DATA_RAW = Path("/content/data/raw")

def load_data(engine="pandas"):
    print(f"[{engine}] Loading shipment data...")
    shipments = pd.read_csv(DATA_RAW / "shipments.csv", parse_dates=["shipment_date"])
    print(f"[{engine}]   {len(shipments):,} shipment rows loaded")
    print(f"[{engine}] Loading country risk data...")
    country_risk = pd.read_csv(DATA_RAW / "country_risk.csv", parse_dates=["month"])
    print(f"[{engine}]   {len(country_risk):,} country-month rows loaded")
    return shipments, country_risk

def engineer_features(shipments, country_risk, engine="pandas"):
    print(f"[{engine}] Step 1/5 - Cleaning rows...")
    before = len(shipments)
    shipments = shipments.dropna(subset=["buyer_id", "buyer_country", "hs_code", "invoice_value_usd"])
    print(f"[{engine}]   Dropped {before - len(shipments):,} rows")
    shipments["buyer_country"] = shipments["buyer_country"].str.strip().str.upper()
    country_risk["country"] = country_risk["country"].str.strip().str.upper()
    shipments["shipment_month"] = shipments["shipment_date"].dt.to_period("M").dt.to_timestamp()

    print(f"[{engine}] Step 2/5 - Joining country risk...")
    joined = shipments.merge(
        country_risk,
        left_on=["buyer_country", "shipment_month"],
        right_on=["country", "month"],
        how="left",
    )
    for c in ["political_stability_score", "currency_volatility_index"]:
        joined[c] = joined[c].fillna(joined[c].median())
    joined["trade_sanctions_flag"] = joined["trade_sanctions_flag"].fillna(0).astype(int)

    print(f"[{engine}] Step 3/5 - Computing buyer-level aggregates...")
    joined = joined.sort_values(["buyer_id", "shipment_date"]).reset_index(drop=True)
    buyer_agg = joined.groupby("buyer_id").agg(
        buyer_total_orders=("shipment_id", "count"),
        buyer_total_value=("invoice_value_usd", "sum"),
        buyer_avg_delay=("payment_delay_days", "mean"),
        buyer_dispute_rate=("was_disputed", "mean"),
        buyer_avg_invoice=("invoice_value_usd", "mean"),
        buyer_paid_in_full_rate=("was_paid_in_full", "mean"),
    ).reset_index()
    joined = joined.merge(buyer_agg, on="buyer_id", how="left")
    joined["buyer_order_rank"] = joined.groupby("buyer_id").cumcount(ascending=False) + 1

    print(f"[{engine}] Step 4/5 - Deriving payment_default_risk label...")
    delay_score = np.clip(joined["payment_delay_days"] / 180.0, 0, 1)
    dispute_penalty = joined["was_disputed"].astype(float) * 0.5
    nonpayment_penalty = (1 - joined["was_paid_in_full"].astype(float)) * 0.3
    joined["payment_default_risk_score"] = 0.3 * delay_score + 0.4 * dispute_penalty + 0.3 * nonpayment_penalty
    joined["payment_default_risk"] = (joined["payment_default_risk_score"] > 0.3).astype(int)

    print(f"[{engine}] Step 5/5 - Dropping temporary columns...")
    joined = joined.drop(columns=[c for c in ["shipment_month", "country", "month"] if c in joined.columns], errors="ignore")

    print(f"[{engine}] Done. Output shape: {joined.shape}")
    return joined

def run_pipeline(engine="pandas"):
    shipments, country_risk = load_data(engine)
    result = engineer_features(shipments, country_risk, engine)
    return result

print("transform.py written to /content/transform.py")

In [ ]:
# ── Step 4: Run CPU (pandas) benchmark ────────────────────
import sys, time, json
sys.path.insert(0, "/content")
from transform import load_data, engineer_features

print("=" * 50)
print("BENCHMARK: CPU (pandas)")
print("=" * 50)

t0 = time.time()
ship, cr = load_data("pandas")
result_cpu = engineer_features(ship, cr, "pandas")
cpu_time = time.time() - t0

print(f"\nCPU TOTAL TIME: {cpu_time:.2f}s")
if cpu_time >= 60:
    print(f"  = {int(cpu_time//60)}m {int(cpu_time%60)}s")

In [ ]:
# ── Step 5: Activate cudf.pandas (GPU backend) ────────────
import cudf.pandas
cudf.pandas.install()

# Verify GPU is active
import pandas as pd
test = pd.DataFrame({"a": [1, 2, 3]})
gpu_active = hasattr(test, "_using_cudf") and test._using_cudf
print(f"GPU active: {gpu_active}")

In [ ]:
# ── Step 6: Run GPU (cudf.pandas) benchmark ──────────────
# Note: After cudf.pandas.install(), pandas imports are GPU-backed.
# We need to reload the module to pick up the GPU path.

import importlib
import transform
importlib.reload(transform)

from transform import load_data, engineer_features

print("=" * 50)
print("BENCHMARK: GPU (cudf.pandas)")
print("=" * 50)

t0 = time.time()
ship, cr = load_data("cudf.pandas")
result_gpu = engineer_features(ship, cr, "cudf.pandas")
gpu_time = time.time() - t0

print(f"\nGPU TOTAL TIME: {gpu_time:.2f}s")

In [ ]:
# ── Step 7: Compare results ───────────────────────────────
print("=" * 50)
print("RESULTS")
print("=" * 50)
print(f"  CPU (pandas):      {cpu_time:.2f}s")
print(f"  GPU (cudf.pandas): {gpu_time:.2f}s")
speedup = cpu_time / gpu_time if gpu_time > 0 else 0
print(f"  Speedup:           {speedup:.1f}x")
print()
print(f"  CPU rows/sec:      {len(result_cpu) / cpu_time:,.0f}")
print(f"  GPU rows/sec:      {len(result_gpu) / gpu_time:,.0f}")
print()

if speedup >= 3:
    print("  VERDICT: Significant GPU acceleration confirmed! ")
elif speedup >= 1.5:
    print("  VERDICT: Moderate GPU speedup.")
else:
    print("  VERDICT: Similar performance. Try with more data (3M+ rows).")

# Save results
results = {
    "engine": {
        "pandas": {
            "total_time_seconds": round(cpu_time, 3),
            "gpu_active": False,
        },
        "cudf.pandas": {
            "total_time_seconds": round(gpu_time, 3),
            "gpu_active": True,
        },
    },
    "speedup": round(speedup, 1),
    "dataset_rows": len(result_cpu),
}
with open("/content/benchmark_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nResults saved to /content/benchmark_results.json")